# Capítulo 12: Análisis de Marketing con IA Generativa

> *«La IA generativa es como un analista junior brillante: te da ideas en segundos, pero necesitas verificar todo antes de gastarte el presupuesto.»*

Este notebook acompaña al Capítulo 12 del libro *Ciencia de Datos sin Filtros*. Aquí implementaremos análisis de marketing digital con IA generativa, usando un dataset realista de 24 meses de campañas.

**Dataset:** Campañas de marketing digital B2B SaaS (728 filas, 5 canales, 24 meses)

**Contenido:**
1. Carga y exploración de datos de campañas
2. Análisis de rendimiento por canal
3. Cálculo de ROAS y CPL
4. Análisis con prompts de IA
5. Recomendaciones de reasignación de presupuesto
6. Visualizaciones ejecutivas
7. Ética: sesgo en datos de marketing

---
## Celda 1: Importaciones

Importamos las librerías necesarias para análisis de marketing y IA.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Librerías cargadas correctamente')
print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

---
## Celda 2: Carga de Datos de Campañas

Cargamos el dataset de 24 meses de campañas de marketing digital.

**Columnas clave:**
- `spend_usd`: Gasto semanal en USD
- `leads`: Leads generados
- `attributed_revenue_usd`: Ingresos atribuidos al canal
- `roas`: Return on Ad Spend (revenue / spend)
- `cpl_usd`: Cost Per Lead (spend / leads)

In [ ]:
# Carga del dataset de campañas
df = pd.read_csv('../datos/datos_marketing_campañas.csv')

print(f'Dimensiones: {df.shape}')
print(f'Columnas: {df.columns.tolist()}')
print(f'Período: {df["week"].min()} a {df["week"].max()}')
print(f'Canales: {df["channel"].unique().tolist()}')
print(f'Regiones: {df["region"].unique().tolist()}')
print()
df.head(10)

In [ ]:
# Información general del dataset
print('INFORMACIÓN GENERAL')
print('=' * 60)
print(f'\nDimensiones: {df.shape}')
print(f'\nTipos de datos:')
print(df.dtypes)
print(f'\nValores nulos:')
print(df.isnull().sum())
print(f'\nDuplicados: {df.duplicated().sum()}')

---
## Celda 3: Análisis de Rendimiento por Canal

El primer paso antes de cualquier recomendación de IA es entender qué está pasando. El **análisis exploratorio de datos (EDA)** es el paso más importante — y el más ignorado.

> 🔍 **Metáfora:** El EDA es como hacer un chequeo médico antes de operar. Saltarse este paso es como operar sin diagnosticar.

In [ ]:
# Resumen de rendimiento por canal
resumen_canal = df.groupby('channel').agg({
    'spend_usd': ['sum', 'mean', 'std'],
    'leads': ['sum', 'mean'],
    'attributed_revenue_usd': ['sum', 'mean'],
    'roas': ['mean', 'median'],
    'cpl_usd': ['mean', 'median']
}).round(2)

print('RENDIMIENTO POR CANAL (24 meses)')
print('=' * 80)
resumen_canal

In [ ]:
# Distribución del gasto por canal
gasto_por_canal = df.groupby('channel')['spend_usd'].sum().sort_values(ascending=False)
total_gasto = gasto_por_canal.sum()

print('DISTRIBUCIÓN DE INVERSIÓN')
print('=' * 50)
for canal, gasto in gasto_por_canal.items():
    pct = (gasto / total_gasto) * 100
    barra = '█' * int(pct / 2)
    print(f'{canal:12s} | ${gasto:>10,.2f} | {pct:>5.1f}% {barra}')
print(f'\n{"TOTAL":12s} | ${total_gasto:>10,.2f} | 100.0%')

In [ ]:
# Tendencia mensual de gasto y leads
df['month'] = pd.to_datetime(df['week']).dt.to_period('M')
tendencia_mensual = df.groupby(['month', 'channel'])['spend_usd'].sum().unstack()

print('TENDENCIA MENSUAL DE GASTO POR CANAL')
print('=' * 60)
tendencia_mensual.tail(6)

---
## Celda 4: Cálculo de ROAS y CPL

### ROAS (Return on Ad Spend)
> 🔍 **Metáfora:** ROAS es "cuánto ganas por cada dólar invertido". Si inviertes $1 y te regresan $4, tu ROAS es 4.0.

**Fórmula:** $ROAS = \frac{\text{Ingresos Atribuidos}}{\text{Gasto en Publicidad}}$

### CPL (Cost Per Lead)
> 🔍 **Metáfora:** CPL es "cuánto te cuesta comprar un cliente potencial".

**Fórmula:** $CPL = \frac{\text{Gasto en Publicidad}}{\text{Leads Generados}}$

In [ ]:
# Cálculo de ROAS por canal
roas_por_canal = df.groupby('channel').agg({
    'spend_usd': 'sum',
    'attributed_revenue_usd': 'sum'
})
roas_por_canal['roas'] = (roas_por_canal['attributed_revenue_usd'] / roas_por_canal['spend_usd']).round(2)
roas_por_canal['ganancia_neta'] = (roas_por_canal['attributed_revenue_usd'] - roas_por_canal['spend_usd']).round(2)
roas_por_canal = roas_por_canal.sort_values('roas', ascending=False)

print('ROAS POR CANAL (Return on Ad Spend)')
print('=' * 70)
for canal, row in roas_por_canal.iterrows():
    print(f'{canal:12s} | Inversión: ${row["spend_usd"]:>10,.2f} | Retorno: ${row["attributed_revenue_usd"]:>10,.2f} | ROAS: {row["roas"]:>6.2f} | Ganancia neta: ${row["ganancia_neta"]:>+10,.2f}')

In [ ]:
# Cálculo de CPL por canal
cpl_por_canal = df.groupby('channel').agg({
    'spend_usd': 'sum',
    'leads': 'sum'
})
cpl_por_canal['cpl'] = (cpl_por_canal['spend_usd'] / cpl_por_canal['leads']).round(2)
cpl_por_canal = cpl_por_canal.sort_values('cpl')

print('CPL POR CANAL (Cost Per Lead)')
print('=' * 60)
for canal, row in cpl_por_canal.iterrows():
    print(f'{canal:12s} | Gasto: ${row["spend_usd"]:>10,.2f} | Leads: {row["leads"]:>6,.0f} | CPL: ${row["cpl"]:>8,.2f}')

In [ ]:
# Diagrama de decisión: ROAS vs CPL
resumen_decision = df.groupby('channel').agg({
    'spend_usd': 'sum',
    'attributed_revenue_usd': 'sum',
    'leads': 'sum'
}).round(2)

resumen_decision['roas'] = (resumen_decision['attributed_revenue_usd'] / resumen_decision['spend_usd']).round(2)
resumen_decision['cpl'] = (resumen_decision['spend_usd'] / resumen_decision['leads']).round(2)
resumen_decision['ganancia_neta'] = (resumen_decision['attributed_revenue_usd'] - resumen_decision['spend_usd']).round(2)

print('DIAGRAMA DE DECISIÓN: ROAS vs CPL')
print('=' * 70)
print(f'{"Canal":12s} | {"ROAS":>6s} | {"CPL":>8s} | {"Ganancia Neta":>14s} | {"Diagnóstico"}')
print('-' * 70)
for canal, row in resumen_decision.iterrows():
    if row['roas'] > 1 and row['cpl'] < 100:
        diagnostico = '★ ESTRELLA'
    elif row['roas'] > 1 and row['cpl'] >= 100:
        diagnostico = '? OPTIMIZAR'
    elif row['roas'] <= 1 and row['cpl'] < 100:
        diagnostico = '? INVESTIGAR'
    else:
        diagnostico = '✗ CORTE'
    print(f'{canal:12s} | {row["roas"]:>6.2f} | ${row["cpl"]:>7.2f} | ${row["ganancia_neta"]:>+12,.2f} | {diagnostico}')

---
## Celda 5: Análisis con Prompts de IA

La IA generativa es tan buena como tus prompts. Un mal prompt genera insights genéricos e inútiles. Un buen prompt genera análisis accionables.

### Reglas para prompts efectivos:
1. **Sé específico:** Incluye datos concretos, no generalidades
2. **Define el contexto:** Tipo de empresa, mercado, objetivos
3. **Pide hipótesis, no solo recomendaciones**
4. **Incluye restricciones:** Presupuesto, capacidad, riesgos

In [ ]:
# Preparar datos resumidos para prompt
resumen_para_prompt = df.groupby('channel').agg({
    'spend_usd': 'sum',
    'attributed_revenue_usd': 'sum',
    'leads': 'sum',
    'roas': 'mean',
    'cpl_usd': 'mean'
}).round(2)

print('DATOS RESUMEN PARA PROMPT DE IA')
print('=' * 60)
print(resumen_para_prompt)

# Prompt de ejemplo para IA
prompt_ejemplo = """
DATOS DE MARKETING (24 meses, 5 canales):
- SEO: Spend ${:.0f} | Revenue ${:.0f} | ROAS {:.2f} | CPL ${:.0f}
- Paid Search: Spend ${:.0f} | Revenue ${:.0f} | ROAS {:.2f} | CPL ${:.0f}
- Social: Spend ${:.0f} | Revenue ${:.0f} | ROAS {:.2f} | CPL ${:.0f}
- Email: Spend ${:.0f} | Revenue ${:.0f} | ROAS {:.2f} | CPL ${:.0f}
- Display: Spend ${:.0f} | Revenue ${:.0f} | ROAS {:.2f} | CPL ${:.0f}

CONTEXTO: Empresa B2B SaaS, mercado competitivo, presupuesto total ${:.0f}/año.

GENERA:
1. Diagnóstico: ¿Por qué Email tiene ROAS tan alto?
2. Hipótesis: ¿Por qué Display tiene el peor rendimiento?
3. Recomendaciones: 5 acciones específicas para mejorar ROAS general
4. Riesgos: ¿Qué podría salir mal con cada recomendación?
""".format(
    resumen_para_prompt.loc['SEO', 'spend_usd'], resumen_para_prompt.loc['SEO', 'attributed_revenue_usd'], resumen_para_prompt.loc['SEO', 'roas'], resumen_para_prompt.loc['SEO', 'cpl_usd'],
    resumen_para_prompt.loc['Paid Search', 'spend_usd'], resumen_para_prompt.loc['Paid Search', 'attributed_revenue_usd'], resumen_para_prompt.loc['Paid Search', 'roas'], resumen_para_prompt.loc['Paid Search', 'cpl_usd'],
    resumen_para_prompt.loc['Social', 'spend_usd'], resumen_para_prompt.loc['Social', 'attributed_revenue_usd'], resumen_para_prompt.loc['Social', 'roas'], resumen_para_prompt.loc['Social', 'cpl_usd'],
    resumen_para_prompt.loc['Email', 'spend_usd'], resumen_para_prompt.loc['Email', 'attributed_revenue_usd'], resumen_para_prompt.loc['Email', 'roas'], resumen_para_prompt.loc['Email', 'cpl_usd'],
    resumen_para_prompt.loc['Display', 'spend_usd'], resumen_para_prompt.loc['Display', 'attributed_revenue_usd'], resumen_para_prompt.loc['Display', 'roas'], resumen_para_prompt.loc['Display', 'cpl_usd'],
    df['spend_usd'].sum()
)

print('\n' + '=' * 60)
print('PROMPT GENERADO PARA IA GENERATIVA')
print('=' * 60)
print(prompt_ejemplo)

In [ ]:
# Función para evaluar respuestas de IA
def evaluar_respuesta_ia(respuesta_ia, datos_reales):
    """
    Evalúa si las recomendaciones de IA son consistentes con los datos.
    
    La IA generativa puede:
    1. Inventar métricas que no existen en los datos
    2. Ignorar tendencias temporales importantes
    3. Recomendar acciones sin considerar el contexto del negocio
    4. Ser demasiado optimista o pesimista
    """
    
    print('EVALUACIÓN DE RESPUESTA DE IA')
    print('=' * 60)
    
    print('\n1. Verificación de métricas:')
    print('   ¿La IA mencionó cifras exactas o aproximadas?')
    print('   ¿Las cifras coinciden con los datos reales?')
    
    print('\n2. Verificación de tendencias:')
    print('   ¿La IA identificó la estacionalidad?')
    print('   ¿Consideró la tendencia temporal?')
    
    print('\n3. Verificación de contexto:')
    print('   ¿La IA sabe que es B2B SaaS?')
    print('   ¿Consideró la canibalización entre canales?')
    print('   ¿Mencionó riesgos específicos?')
    
    print('\n4. Puntuación de confiabilidad:')
    print('   Si la IA acertó >80%: USAR con precaución')
    print('   Si la IA acertó 50-80%: REVISAR cada recomendación')
    print('   Si la IA acertó <50%: DESCARTAR y hacer análisis manual')

print('EJEMPLO: Evaluación de respuesta de IA')
print('=' * 60)
evaluar_respuesta_ia('respuesta_ejemplo', df)

---
## Celda 6: Recomendaciones de Reasignación

Reasignar presupuesto entre canales no es como mover fichas de ajedrez. Cada canal tiene:
- **Efecto de diminishing returns:** Más inversión no siempre = más resultados
- **Efecto de canibalización:** Un canal puede robar tráfico de otro
- **Efecto de brand awareness:** Display puede no generar leads directos, pero fortalece la marca

In [ ]:
# Modelo de reasignación basado en eficiencia marginal
def calcular_eficiencia_canal(df, canal):
    """Calcula la eficiencia marginal de un canal."""
    datos_canal = df[df['channel'] == canal].sort_values('week')
    
    # ROI promedio
    roi_promedio = datos_canal['attributed_revenue_usd'].sum() / datos_canal['spend_usd'].sum()
    
    # Tendencia (¿está mejorando o empeorando?)
    if len(datos_canal) > 12:
        primer_semestre = datos_canal.head(12)['roas'].mean()
        segundo_semestre = datos_canal.tail(12)['roas'].mean()
        tendencia = 'MEJORANDO' if segundo_semestre > primer_semestre else 'EMPEORANDO'
    else:
        tendencia = 'INSUFICIENTE'
    
    return {
        'canal': canal,
        'roi_promedio': round(roi_promedio, 2),
        'tendencia': tendencia,
        'roas_reciente': round(datos_canal.tail(4)['roas'].mean(), 2)
    }

# Evaluar cada canal
canales = df['channel'].unique()
evaluacion = [calcular_eficiencia_canal(df, c) for c in canales]
evaluacion_df = pd.DataFrame(evaluacion).sort_values('roi_promedio', ascending=False)

print('EVALUACIÓN DE EFICIENCIA POR CANAL')
print('=' * 65)
print(evaluacion_df.to_string(index=False))

In [ ]:
# Recomendación basada en ROI y tendencia
print('\n' + '=' * 70)
print('RECOMENDACIÓN DE REASIGNACIÓN DE PRESUPUESTO')
print('=' * 70)

for _, row in evaluacion_df.iterrows():
    canal = row['canal']
    roi = row['roi_promedio']
    tendencia = row['tendencia']
    
    if roi > 1 and tendencia == 'MEJORANDO':
        accion = '↑ AUMENTAR inversión (+15-20%)'
    elif roi > 1 and tendencia == 'EMPEORANDO':
        accion = '→ MANTENER y optimizar'
    elif roi < 1 and tendencia == 'MEJORANDO':
        accion = '→ MANTENER con monitoreo intensivo'
    else:
        accion = '↓ REDUCIR inversión (-20-30%)'
    
    print(f'\n{canal}:')
    print(f'  ROI actual: {roi:.2f} | Tendencia: {tendencia}')
    print(f'  Acción recomendada: {accion}')

---
## Celda 7: Visualizaciones Ejecutivas

Las visualizaciones son la forma más efectiva de comunicar hallazgos a tomadores de decisiones. Un gráfico claro vale más que mil filas de datos.

In [ ]:
# Verificar si matplotlib está disponible
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['figure.figsize'] = (12, 6)
    matplotlib.rcParams['font.size'] = 11
    
    print('Matplotlib disponible. Generando visualizaciones...')
    
    # 1. ROAS por canal
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # ROAS por canal
    roas_data = df.groupby('channel')['roas'].mean().sort_values(ascending=True)
    colors = ['#ff6b6b' if x < 1 else '#51cf66' for x in roas_data.values]
    axes[0, 0].barh(roas_data.index, roas_data.values, color=colors)
    axes[0, 0].axvline(x=1, color='red', linestyle='--', alpha=0.7, label='Break-even (ROAS=1)')
    axes[0, 0].set_title('ROAS Promedio por Canal', fontweight='bold')
    axes[0, 0].set_xlabel('ROAS')
    axes[0, 0].legend()
    
    # CPL por canal
    cpl_data = df.groupby('channel')['cpl_usd'].mean().sort_values(ascending=True)
    axes[0, 1].barh(cpl_data.index, cpl_data.values, color='#339af0')
    axes[0, 1].set_title('CPL Promedio por Canal', fontweight='bold')
    axes[0, 1].set_xlabel('CPL (USD)')
    
    # Distribución de gasto
    gasto_data = df.groupby('channel')['spend_usd'].sum()
    axes[1, 0].pie(gasto_data.values, labels=gasto_data.index, autopct='%1.1f%%', startangle=90)
    axes[1, 0].set_title('Distribución del Gasto Total', fontweight='bold')
    
    # Tendencia temporal de ROAS
    df_temporal = df.copy()
    df_temporal['month'] = pd.to_datetime(df_temporal['week']).dt.to_period('M').astype(str)
    roas_temporal = df_temporal.groupby(['month', 'channel'])['roas'].mean().unstack()
    roas_temporal.plot(ax=axes[1, 1], marker='o', linewidth=2)
    axes[1, 1].set_title('ROAS Mensual por Canal', fontweight='bold')
    axes[1, 1].set_xlabel('Mes')
    axes[1, 1].set_ylabel('ROAS')
    axes[1, 1].legend(loc='upper right', fontsize=8)
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('../datos/visualizaciones_marketing.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Visualizaciones generadas correctamente.')
    
except ImportError:
    print('Matplotlib no está instalado. Instala con: pip install matplotlib')
    print('\nAlternativa: Usa las tablas de resumen generadas en celdas anteriores.')

---
## Celda 8: Ética - Sesgo en Datos de Marketing

> ⚠️ **¿Qué pasa si incluimos código postal?**

Incluir datos geográficos detallados en análisis de marketing puede parecer inocente, pero los datos geográficos son un **proxy peligroso** para raza, etnia, nivel socioeconómico y acceso a tecnología.

In [ ]:
# Análisis de distribución geográfica
print('DISTRIBUCIÓN GEOGRÁFICA DE INVERSIÓN')
print('=' * 60)
dist_region = df.groupby('region').agg({
    'spend_usd': 'sum',
    'leads': 'sum',
    'attributed_revenue_usd': 'sum',
    'roas': 'mean'
}).round(2)
dist_region['pct_inversion'] = (dist_region['spend_usd'] / dist_region['spend_usd'].sum() * 100).round(1)
dist_region['cpl'] = (dist_region['spend_usd'] / dist_region['leads']).round(2)

print(dist_region)

print('\n' + '=' * 60)
print('PREGUNTA ÉTICA:')
print('¿Estamos invirtiendo más en regiones de altos ingresos')
print('porque ahí hay más "clientes ideales", o porque nuestros')
print('datos de atribución están sesgados hacia esos mercados?')

In [ ]:
# Análisis por segmento de audiencia
segmento_stats = df.groupby('audience_segment').agg({
    'spend_usd': 'sum',
    'leads': 'sum',
    'attributed_revenue_usd': 'sum',
    'roas': 'mean'
}).round(2)

segmento_stats['cpl'] = (segmento_stats['spend_usd'] / segmento_stats['leads']).round(2)
segmento_stats = segmento_stats.sort_values('roas', ascending=False)

print('RENDIMIENTO POR SEGMENTO DE AUDIENCIA')
print('=' * 70)
print(segmento_stats)

print('\n' + '=' * 70)
print('PREGUNTA ÉTICA:')
print('¿Estamos priorizando el segmento "Profesionales 41-55"')
print('porque genera más revenue, o porque nuestro targeting')
print('está sesgado hacia ese perfil demográfico?')

In [ ]:
# Checklist ético para análisis de marketing con IA
checklist_etico = {
    '1. Datos de entrada': [
        '¿Los datos de atribución son confiables?',
        '¿Hay sesgo en la captura de datos?',
        '¿Estamos excluyendo grupos demográficos?',
        '¿Los datos reflejan decisiones previas sesgadas?'
    ],
    '2. Modelo de atribución': [
        '¿El modelo de atribución es justo?',
        '¿Considera la contribución de todos los canales?',
        '¿Penaliza canales de awareness (Display, Social)?',
        '¿Es transparente en su lógica?'
    ],
    '3. Recomendaciones de IA': [
        '¿Las recomendaciones excluyen grupos?',
        '¿Consideran el impacto en la diversidad?',
        '¿Son auditables y explicables?',
        '¿Incluyen métricas de equidad?'
    ],
    '4. Implementación': [
        '¿Monitoreamos impacto en diferentes grupos?',
        '¿Tenemos mecanismos de apelación?',
        '¿Revisamos regularmente el sesgo?',
        '¿Documentamos decisiones?'
    ]
}

print('CHECKLIST ÉTICO PARA MARKETING CON IA')
print('=' * 60)
for seccion, preguntas in checklist_etico.items():
    print(f'\n{seccion}:')
    for i, pregunta in enumerate(preguntas, 1):
        print(f'  {i}. {pregunta}')

In [ ]:
# Las 5 reglas de verificación del Científico de Datos Escéptico
print('LAS 5 REGLAS DE VERIFICACIÓN')
print('=' * 60)
print()
print('1. REGLA DE LA FUENTE:')
print('   ¿De dónde vienen los datos que alimentaron a la IA?')
print('   ¿Son representativos de tu mercado actual?')
print()
print('2. REGLA DE LA TEMPORALIDAD:')
print('   ¿La IA considera que el mercado cambia?')
print('   ¿Qué funcionó hace 12 meses puede no funcionar ahora?')
print()
print('3. REGLA DE LA CANIBALIZACIÓN:')
print('   ¿La IA considera que los canales no operan en vacío?')
print('   ¿Qué pasa con la interacción entre canales?')
print()
print('4. REGLA DEL CONTRAFACTUAL:')
print('   ¿Qué pasaría si haces lo contrario de lo que la IA recomienda?')
print('   ¿Has evaluado esa alternativa?')
print()
print('5. REGLA DE LA AUDITORÍA:')
print('   ¿Puedes explicar por qué la IA tomó esa decisión?')
print('   Si no, no la implementes.')

---
## Resumen del Capítulo 12

### Lecciones Clave:

1. **La IA no reemplaza el análisis exploratorio.** Antes de pedirle a la IA que interprete tus datos, necesitas entenderlos tú mismo.

2. **Los promedios son peligrosos.** Un ROAS promedio de 3.5 puede esconder semanas con ROAS de 0.1 y semanas con ROAS de 15.

3. **La ética no es opcional.** Incluir código postal, edad o género en análisis de marketing tiene consecuencias reales para personas reales.

4. **Los datos de atribución son mentiras convenientes.** Ningún modelo de atribución captura perfectamente la contribución de cada canal.

5. **Verificar es más lento que confiar.** Pero verificar te evita decisiones catastróficas basadas en datos sesgados.

> *«La IA generativa no es un oráculo — es un espejo que refleja los sesgos de tus datos con una confianza que no merece.»*

### Referencias:
- Kumar, V., & Reinartz, W. (2018). *Customer Relationship Management*. Springer.
- Provost, F., & Fawcett, T. (2013). *Data Science for Business*. O'Reilly.
- Turtle, T. (2023). "AI-Powered Marketing Analytics". *Harvard Business Review*.